In [10]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

import utils.data_processing_bronze_table
import utils.data_processing_silver_table
import utils.data_processing_gold_table

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 06:31:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2025-11-01"

# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
print(dates_str_lst)

['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01', '2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01', '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01', '2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01']


In [4]:
# BRONZE
# create bronze datalake - lms
bronze_lms_directory = "datamart/bronze/lms/"

if not os.path.exists(bronze_lms_directory):
    os.makedirs(bronze_lms_directory)

# run bronze backfill - lms
for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_lms_table(date_str, bronze_lms_directory, spark)



2023-01-01row count: 530


saved to: datamart/bronze/lms/bronze_loan_daily_2023_01_01.csv


2023-02-01row count: 1031


saved to: datamart/bronze/lms/bronze_loan_daily_2023_02_01.csv
2023-03-01row count: 1537
saved to: datamart/bronze/lms/bronze_loan_daily_2023_03_01.csv
2023-04-01row count: 2047
saved to: datamart/bronze/lms/bronze_loan_daily_2023_04_01.csv
2023-05-01row count: 2568
saved to: datamart/bronze/lms/bronze_loan_daily_2023_05_01.csv
2023-06-01row count: 3085
saved to: datamart/bronze/lms/bronze_loan_daily_2023_06_01.csv
2023-07-01row count: 3556


saved to: datamart/bronze/lms/bronze_loan_daily_2023_07_01.csv
2023-08-01row count: 4037
saved to: datamart/bronze/lms/bronze_loan_daily_2023_08_01.csv
2023-09-01row count: 4491
saved to: datamart/bronze/lms/bronze_loan_daily_2023_09_01.csv
2023-10-01row count: 4978
saved to: datamart/bronze/lms/bronze_loan_daily_2023_10_01.csv
2023-11-01row count: 5469
saved to: datamart/bronze/lms/bronze_loan_daily_2023_11_01.csv
2023-12-01row count: 5428
saved to: datamart/bronze/lms/bronze_loan_daily_2023_12_01.csv
2024-01-01row count: 5412
saved to: datamart/bronze/lms/bronze_loan_daily_2024_01_01.csv
2024-02-01row count: 5424
saved to: datamart/bronze/lms/bronze_loan_daily_2024_02_01.csv
2024-03-01row count: 5425
saved to: datamart/bronze/lms/bronze_loan_daily_2024_03_01.csv
2024-04-01row count: 5417
saved to: datamart/bronze/lms/bronze_loan_daily_2024_04_01.csv
2024-05-01row count: 5391
saved to: datamart/bronze/lms/bronze_loan_daily_2024_05_01.csv
2024-06-01row count: 5418
saved to: datamart/br

NameError: name 'source' is not defined

In [12]:
# bronze - financials and attributes
table_dict = {
    'source': ['data/features_attributes.csv', 'data/features_financials.csv'],
    'directory': ['datamart/bronze/attributes/', 'datamart/bronze/financials/'],
    'filename': ['cust_attr', 'cust_fin']
             }

for i in range(len(table_dict['source'])):
    if not os.path.exists(table_dict['directory'][i]):
        os.makedirs(table_dict['directory'][i])
    utils.data_processing_bronze_table.process_bronze_other_tables(
        table_dict['source'][i], 
        table_dict['directory'][i], 
        table_dict['filename'][i], 
        spark
    )

features_attributes.csv has 12500 rows.
saved to:  datamart/bronze/attributes/bronze_cust_attr.csv
features_financials.csv has 12500 rows.
saved to:  datamart/bronze/financials/bronze_cust_fin.csv


In [14]:
import importlib
import utils.data_processing_bronze_table

importlib.reload(utils.data_processing_bronze_table)

# bronze - clickstream
bronze_clickstream_directory = "datamart/bronze/clickstream/"

if not os.path.exists(bronze_clickstream_directory):
    os.makedirs(bronze_clickstream_directory)

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"
dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
print(dates_str_lst)

for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_clickstream_table(
        date_str, 
        bronze_clickstream_directory, 
        spark
    )

['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01', '2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01', '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01']
2023-01-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2023_01_01.csv
2023-02-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2023_02_01.csv
2023-03-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2023_03_01.csv
2023-04-01row count: 8974


saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2023_04_01.csv
2023-05-01row count: 8974


saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2023_05_01.csv
2023-06-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2023_06_01.csv
2023-07-01row count: 8974


saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2023_07_01.csv


2023-08-01row count: 8974


...

saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2024_09_01.csv
2024-10-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2024_10_01.csv


2024-11-01row count: 8974


saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2024_11_01.csv


2024-12-01row count: 8974


saved to: datamart/bronze/clickstream/bronze_clickstream_daily_2024_12_01.csv
